# Hybrid Physics-ML Reactor — Production Pipeline

**How to use this notebook**

| Goal | Cells to run |
|---|---|
| First time / retrain | Run **all cells** top to bottom |
| Predict a new reactor state | Run **Cell 1** (imports) → **Cell 7** (load & predict) |
| Change profit parameters | Edit **Cell 1** constants → re-run **Cell 6** onward |

Trained artefacts are saved to `models/` so you never need to retrain unless your data changes.

In [9]:
# ════════════════════════════════════════════════════════════════════════
# CELL 1 — CONSTANTS & IMPORTS
# Edit this cell to change physical parameters or cost assumptions.
# Everything else reads from these variables — change here, nowhere else.
# ════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import joblib, json, os, warnings
from datetime import datetime, timezone
from scipy.integrate import odeint
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Physical constants ────────────────────────────────────────────────
R    = 8.314
Ea1, Ea2, Ea3 = 50_000, 70_000, 60_000   # J/mol
A1, A2, A3    = 5.1e7,  2.0e6,  1.5e7
T_NOMINAL     = 300.0   # K — ideal model temperature
TEMP_AMP      = 25.0    # K — day/night swing
K_DEACT       = 0.05    # catalyst deactivation rate

# ── Simulation grid ───────────────────────────────────────────────────
T_MAX, N_POINTS = 48, 1_000
RANDOM_SEED     = 42

# ── Feature order (must never change after training) ─────────────────
FEATURES = ['time', 'temp', 'temp_sq', 'ideal_B', 'B_lag1', 'B_lag2', 'activity']

# ── Profit function parameters ────────────────────────────────────────
PRICE, OP_COST, CATALYST_COST = 1_000, 20, 50

# ── Paths ─────────────────────────────────────────────────────────────
MODEL_PATH    = 'models/best_model.pkl'
SCALER_PATH   = 'models/scaler.pkl'
METADATA_PATH = 'models/metadata.json'
os.makedirs('models', exist_ok=True)

print('✓ Constants loaded')

✓ Constants loaded


In [4]:
# ════════════════════════════════════════════════════════════════════════
# CELL 2 — PHYSICS (ODE functions)
# These are pure functions — no side effects, easy to unit-test.
# ════════════════════════════════════════════════════════════════════════

def ideal_kinetics(y, t):
    """Ideal reactor: constant T=300K, perfect catalyst."""
    A, B, C, D = y
    k1 = A1 * np.exp(-Ea1 / (R * T_NOMINAL))
    k2 = A2 * np.exp(-Ea2 / (R * T_NOMINAL))
    k3 = A3 * np.exp(-Ea3 / (R * T_NOMINAL))
    return [-(k1+k3)*A,  k1*A - k2*B,  k2*B,  k3*A]


def real_system(y, t):
    """Industrial reactor: sinusoidal temperature + catalyst deactivation."""
    A, B, C, D, activity = y
    T  = T_NOMINAL + TEMP_AMP * np.sin(2 * np.pi * t / 24)
    k1 = A1 * np.exp(-Ea1 / (R * T)) * activity
    k2 = A2 * np.exp(-Ea2 / (R * T)) * activity
    k3 = A3 * np.exp(-Ea3 / (R * T)) * activity
    return [-(k1+k3)*A,  k1*A - k2*B,  k2*B,  k3*A,  -K_DEACT*activity*A]


def simulate(t_grid):
    """Solve both ODEs. Returns (ideal_sol [N,4], real_sol [N,5])."""
    ideal = odeint(ideal_kinetics, [1, 0, 0, 0],    t_grid)
    real  = odeint(real_system,    [1, 0, 0, 0, 1], t_grid)
    return ideal, real


print('✓ Physics functions defined')

✓ Physics functions defined


In [5]:
# ════════════════════════════════════════════════════════════════════════
# CELL 3 — DATA GENERATION & PREPROCESSING
# Runs the simulation, adds sensor noise + missing data, builds features.
# ════════════════════════════════════════════════════════════════════════

np.random.seed(RANDOM_SEED)
t = np.linspace(0, T_MAX, N_POINTS)

ideal_sol, real_sol = simulate(t)

df = pd.DataFrame({
    'time':     t,
    'ideal_B':  ideal_sol[:, 1],
    'real_B':   real_sol[:,  1],
    'temp':     T_NOMINAL + TEMP_AMP * np.sin(2 * np.pi * t / 24),
    'activity': real_sol[:, 4],
})

# Sensor noise + random outages
df['real_B'] += np.random.normal(0, 0.002, len(df))
df.loc[df.sample(frac=0.05, random_state=0).index, 'real_B'] = np.nan

valid_mask = ~df['real_B'].isna()   # rows with real measurements

# Imputation + feature engineering
df['real_B_imputed'] = df['real_B'].interpolate(method='linear').bfill()
df['B_lag1']         = df['real_B_imputed'].shift(1).bfill()
df['B_lag2']         = df['real_B_imputed'].shift(2).bfill()
df['temp_sq']        = df['temp'] ** 2
df['residual']       = df['real_B_imputed'] - df['ideal_B']   # ML target

print(f'✓ Dataset ready  — {len(df)} rows, {valid_mask.sum()} with real measurements')
df[FEATURES + ['residual']].describe().round(4)

✓ Dataset ready  — 1000 rows, 950 with real measurements


,time,temp,temp_sq,ideal_B,B_lag1,B_lag2,activity,residual
count,1000.0000,1000.0000,1000.0000,1000.0000,1000.0000,1000.0000,1000.0000,1000.0000
mean,24.0000,300.0000,90312.1875,0.7906,0.8952,0.8942,0.8131,0.1056
std,13.8772,17.6777,10608.9079,0.2469,0.2096,0.2115,0.0378,0.1101
min,0.0000,275.0000,75625.0170,0.0000,0.0010,0.0010,0.7914,0.0003
25%,12.0000,282.3502,79721.6160,0.6982,0.9482,0.9478,0.7921,0.0208
50%,24.0000,300.0000,90000.0000,0.9063,0.9626,0.9625,0.7975,0.0478
75%,36.0000,317.6498,100901.4186,0.9683,0.9906,0.9906,0.8157,0.1736
max,48.0000,325.0000,105624.9799,0.9868,0.9970,0.9970,1.0000,0.3553


In [6]:
# ════════════════════════════════════════════════════════════════════════
# CELL 4 — TRAIN / TEST SPLIT & SCALING
# Scaler is fit on training data only (no leakage).
# ════════════════════════════════════════════════════════════════════════

X_all   = df[FEATURES].values
y_all   = df['residual'].values

X_valid = X_all[valid_mask]
y_valid = y_all[valid_mask]

split    = int(len(X_valid) * 0.80)
X_train, y_train = X_valid[:split], y_valid[:split]
X_test,  y_test  = X_valid[split:], y_valid[split:]

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
X_all_s   = scaler.transform(X_all)        # for full-trajectory hybrid predictions

print(f'✓ Split complete  — train: {X_train_s.shape}  test: {X_test_s.shape}')

✓ Split complete  — train: (760, 7)  test: (190, 7)


In [7]:
# ════════════════════════════════════════════════════════════════════════
# CELL 5 — TRAINING
# Trains all three models. Reduce n_trials to speed up during development.
# ════════════════════════════════════════════════════════════════════════

# ── XGBoost with Optuna tuning ─────────────────────────────────────────
def xgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators',   100, 600),
        'max_depth':        trial.suggest_int('max_depth',         3,   8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample',       0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree',0.6, 1.0),
        'random_state': 42, 'verbosity': 0,
    }
    maes = []
    for tr, val in TimeSeriesSplit(n_splits=5).split(X_train_s):
        m = xgb.XGBRegressor(**params)
        m.fit(X_train_s[tr], y_train[tr])
        maes.append(mean_absolute_error(y_train[val], m.predict(X_train_s[val])))
    return np.mean(maes)

print('Tuning XGBoost...')
study = optuna.create_study(direction='minimize')
study.optimize(xgb_objective, n_trials=50)
xgb_model = xgb.XGBRegressor(**study.best_params, random_state=42, verbosity=0)
xgb_model.fit(X_train_s, y_train)
print(f'  Best params: {study.best_params}')

# ── LightGBM ───────────────────────────────────────────────────────────
print('Training LightGBM...')
X_train_df = pd.DataFrame(X_train_s, columns=FEATURES)
lgb_model = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, max_depth=5,
                                random_state=42, verbose=-1)
lgb_model.fit(X_train_df, y_train)

# ── RandomForest ───────────────────────────────────────────────────────
print('Training RandomForest...')
rf_model = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42)
rf_model.fit(X_train_s, y_train)

print('\n✓ All models trained')

Tuning XGBoost...
  Best params: {'n_estimators': 286, 'max_depth': 3, 'learning_rate': 0.06780340858024224, 'subsample': 0.634131398027476, 'colsample_bytree': 0.7960411529008847}
Training LightGBM...
Training RandomForest...

✓ All models trained


In [8]:
# ════════════════════════════════════════════════════════════════════════
# CELL 6 — EVALUATE, SELECT BEST & SAVE
# Picks the winner by active-region R², saves model + scaler + metadata.
# ════════════════════════════════════════════════════════════════════════

# ── Hybrid predictions on full dataset ────────────────────────────────
X_all_df = pd.DataFrame(X_all_s, columns=FEATURES)

df['hybrid_xgb'] = df['ideal_B'] + xgb_model.predict(X_all_s)
df['hybrid_lgb'] = df['ideal_B'] + lgb_model.predict(X_all_df)
df['hybrid_rf']  = df['ideal_B'] + rf_model.predict(X_all_s)

# ── Evaluate on active region (where B actually matters) ───────────────
active      = df['ideal_B'] > 0.01
real_active = df.loc[active, 'real_B_imputed'].values

def score(y_true, y_pred):
    return {
        'mae':  round(mean_absolute_error(y_true, y_pred), 5),
        'rmse': round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 5),
        'r2':   round(r2_score(y_true, y_pred), 5),
    }

all_scores = {
    'XGBoost':      score(real_active, df.loc[active, 'hybrid_xgb'].values),
    'LightGBM':     score(real_active, df.loc[active, 'hybrid_lgb'].values),
    'RandomForest': score(real_active, df.loc[active, 'hybrid_rf'].values),
}

print('\n── Model performance (active region: ideal_B > 0.01) ──')
for name, s in all_scores.items():
    marker = '  ← BEST' if s['r2'] == max(v['r2'] for v in all_scores.values()) else ''
    print(f"  {name:<15} MAE={s['mae']}  RMSE={s['rmse']}  R²={s['r2']}{marker}")

# ── Select winner ─────────────────────────────────────────────────────
best_name  = max(all_scores, key=lambda n: all_scores[n]['r2'])
best_model = {'XGBoost': xgb_model, 'LightGBM': lgb_model, 'RandomForest': rf_model}[best_name]
best_col   = {'XGBoost': 'hybrid_xgb', 'LightGBM': 'hybrid_lgb', 'RandomForest': 'hybrid_rf'}[best_name]

# ── Profit optimisation ────────────────────────────────────────────────
def profit(yield_b, t_arr):
    return (yield_b * PRICE) - (t_arr * OP_COST) - CATALYST_COST

df['profit_hybrid'] = profit(df[best_col],     df['time'])
df['profit_ideal']  = profit(df['ideal_B'],    df['time'])

opt_time   = df.loc[df['profit_hybrid'].idxmax(), 'time']
ideal_time = df.loc[df['profit_ideal'].idxmax(),  'time']

print(f'\n── Decision report ───────────────────────────────────')
print(f'  Hybrid stop time : {opt_time:.2f} h')
print(f'  Ideal  stop time : {ideal_time:.2f} h')
print(f'  Improvement      : {abs(opt_time - ideal_time):.2f} h')

# ── Save artefacts ─────────────────────────────────────────────────────
joblib.dump(best_model, MODEL_PATH,  compress=3)
joblib.dump(scaler,     SCALER_PATH, compress=3)

metadata = {
    'model_name':    best_name,
    'trained_at':    datetime.now(timezone.utc).isoformat(),
    'features':      FEATURES,
    'metrics':       all_scores[best_name],
    'all_scores':    all_scores,
    'profit_params': {'price': PRICE, 'op_cost': OP_COST, 'catalyst_cost': CATALYST_COST},
    'decision':      {'hybrid_stop_h': round(opt_time, 2),
                      'ideal_stop_h':  round(ideal_time, 2),
                      'improvement_h': round(abs(opt_time - ideal_time), 2)},
}
with open(METADATA_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

# ── Smoke test — reload and verify ────────────────────────────────────
_m = joblib.load(MODEL_PATH)
_s = joblib.load(SCALER_PATH)
_x = _s.transform(X_all[:5])
_inp = pd.DataFrame(_x, columns=FEATURES) if best_name == 'LightGBM' else _x
assert np.allclose(_m.predict(_inp), best_model.predict(_inp), atol=1e-6), \
    'Smoke test FAILED — reloaded model differs!'

print(f'\n✓ Saved  {MODEL_PATH}  ({os.path.getsize(MODEL_PATH)/1024:.0f} KB)')
print(f'✓ Saved  {SCALER_PATH}')
print(f'✓ Saved  {METADATA_PATH}')
print('✓ Smoke test passed')


── Model performance (active region: ideal_B > 0.01) ──
  XGBoost         MAE=0.0028  RMSE=0.00464  R²=0.99947  ← BEST
  LightGBM        MAE=0.00502  RMSE=0.01006  R²=0.99752
  RandomForest    MAE=0.0028  RMSE=0.00472  R²=0.99945

── Decision report ───────────────────────────────────
  Hybrid stop time : 8.60 h
  Ideal  stop time : 16.00 h
  Improvement      : 7.40 h

✓ Saved  models/best_model.pkl  (55 KB)
✓ Saved  models/scaler.pkl
✓ Saved  models/metadata.json
✓ Smoke test passed


In [10]:
# ════════════════════════════════════════════════════════════════════════
# CELL 7 — PREDICT  (run this cell anytime, no retraining needed)
#
# Two modes:
#   predict_point()      — single reactor state → hybrid B + correction
#   predict_trajectory() — full 48h run → B profile + optimal stop time
# ════════════════════════════════════════════════════════════════════════

import joblib, json, numpy as np, pandas as pd

# ── Load from disk (works even in a fresh kernel) ─────────────────────
_model  = joblib.load(MODEL_PATH)
_scaler = joblib.load(SCALER_PATH)
with open(METADATA_PATH) as f:
    _meta = json.load(f)

def _scale_and_predict(X_raw):
    """Scale features and call the right model input type."""
    X_s = _scaler.transform(np.atleast_2d(X_raw))
    if _meta['model_name'] == 'LightGBM':
        return _model.predict(pd.DataFrame(X_s, columns=_meta['features']))
    return _model.predict(X_s)


def predict_point(time_h, temp_K, ideal_B, B_lag1, B_lag2, activity):
    """
    Predict hybrid B for a single reactor state.

    Parameters
    ----------
    time_h   : current elapsed time (hours)
    temp_K   : current reactor temperature (Kelvin)
    ideal_B  : ideal ODE prediction for B at this state
    B_lag1   : B measurement from 1 time step ago
    B_lag2   : B measurement from 2 time steps ago
    activity : current catalyst activity (0–1)

    Returns dict with hybrid_B, ideal_B, ml_correction
    """
    features = [time_h, temp_K, temp_K**2, ideal_B, B_lag1, B_lag2, activity]
    correction = float(_scale_and_predict([features])[0])
    return {
        'hybrid_B':      round(ideal_B + correction, 6),
        'ideal_B':       round(ideal_B, 6),
        'ml_correction': round(correction, 6),
        'model_used':    _meta['model_name'],
    }


def predict_trajectory(n_points=500, t_max_h=48.0):
    """
    Predict the full B trajectory and find the profit-optimal stop time.
    Uses the ideal ODE + real activity from the simulation.

    Returns a DataFrame with columns:
        time, ideal_B, hybrid_B, profit, is_optimal_stop
    """
    from scipy.integrate import odeint

    t_grid   = np.linspace(0, t_max_h, n_points)
    ideal    = odeint(ideal_kinetics, [1, 0, 0, 0],    t_grid)
    real     = odeint(real_system,    [1, 0, 0, 0, 1], t_grid)

    ideal_B  = ideal[:, 1]
    activity = real[:, 4]
    temp     = T_NOMINAL + TEMP_AMP * np.sin(2 * np.pi * t_grid / 24)
    B_lag1   = np.concatenate([[ideal_B[0]], ideal_B[:-1]])
    B_lag2   = np.concatenate([[ideal_B[0], ideal_B[0]], ideal_B[:-2]])

    X_raw = np.column_stack([t_grid, temp, temp**2, ideal_B, B_lag1, B_lag2, activity])
    hybrid_B = ideal_B + _scale_and_predict(X_raw)

    profits  = (hybrid_B * PRICE) - (t_grid * OP_COST) - CATALYST_COST
    best_idx = int(np.argmax(profits))

    result = pd.DataFrame({
        'time':             t_grid,
        'ideal_B':          ideal_B,
        'hybrid_B':         hybrid_B,
        'profit':           profits,
        'is_optimal_stop':  [i == best_idx for i in range(n_points)],
    })

    stop_time = t_grid[best_idx]
    print(f'✓ Trajectory computed  ({n_points} points, horizon={t_max_h}h)')
    print(f'  Optimal stop time : {stop_time:.2f} h')
    print(f'  Max profit        : {profits[best_idx]:,.1f}')
    print(f'  hybrid_B at stop  : {hybrid_B[best_idx]:.4f}')
    return result


print(f'✓ Predictor ready  ({_meta["model_name"]} — R²={_meta["metrics"]["r2"]})')
print()
print('  Usage:')
print('    predict_point(time_h=10, temp_K=302, ideal_B=0.50, B_lag1=0.49, B_lag2=0.48, activity=0.85)')
print('    predict_trajectory()')

✓ Predictor ready  (XGBoost — R²=0.99947)

  Usage:
    predict_point(time_h=10, temp_K=302, ideal_B=0.50, B_lag1=0.49, B_lag2=0.48, activity=0.85)
    predict_trajectory()


In [11]:
# ════════════════════════════════════════════════════════════════════════
# CELL 8 — EXAMPLE PREDICTIONS  (edit inputs here)
# ════════════════════════════════════════════════════════════════════════

# ── Point prediction ──────────────────────────────────────────────────
# Change these values to match your actual reactor readings:
result = predict_point(
    time_h   = 10.0,    # hours elapsed
    temp_K   = 302.0,   # current temperature (K)
    ideal_B  = 0.50,    # what the ideal ODE says B should be
    B_lag1   = 0.49,    # B one reading ago
    B_lag2   = 0.48,    # B two readings ago
    activity = 0.85,    # catalyst activity (measure or estimate)
)
print('Point prediction:')
for k, v in result.items():
    print(f'  {k:<18} {v}')

print()

# ── Full trajectory + optimal stop ────────────────────────────────────
traj = predict_trajectory(n_points=500, t_max_h=48.0)

print()
print('Trajectory sample (every 50th row):')
traj.iloc[::50][['time','ideal_B','hybrid_B','profit','is_optimal_stop']].round(4)

Point prediction:
  hybrid_B           0.766473
  ideal_B            0.5
  ml_correction      0.266473
  model_used         XGBoost

✓ Trajectory computed  (500 points, horizon=48.0h)
  Optimal stop time : 9.72 h
  Max profit        : 689.9
  hybrid_B at stop  : 0.9342

Trajectory sample (every 50th row):


,time,ideal_B,hybrid_B,profit,is_optimal_stop
0,0.0000,0.0000,0.0012,-48.7834,False
50,4.8096,0.3824,0.6500,503.8184,False
100,9.6192,0.6178,0.9306,688.1739,False
150,14.4289,0.7626,0.9639,625.3488,False
200,19.2385,0.8518,0.9608,526.0494,False
250,24.0481,0.9067,0.9680,437.0839,False
300,28.8577,0.9405,0.9919,364.7880,False
350,33.6673,0.9613,0.9943,270.9858,False
400,38.4770,0.9741,0.9952,175.6496,False
450,43.2866,0.9820,1.0058,90.0404,False
